# Demo: What is RAG & Why it Matters
### Module 5, Topic 1 — RAG from Scratch

**What you'll see in this notebook:**
1. Ask an LLM a question it has no way of knowing the answer to
2. Watch it either refuse or hallucinate an answer
3. Manually supply the relevant text as context
4. Ask the same question again — this time it answers correctly

This is the entire idea behind RAG, done by hand once so you understand exactly what the rest of this module automates.


## Step 0 — Install the Anthropic SDK

We'll use Claude for this demo. Run the cell below once.

In [ ]:
!pip install anthropic --quiet

## Step 1 — Set Up the Client

Your API key should be stored as an environment variable — never hardcode it directly in a notebook.

In [ ]:
import os
import anthropic

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
MODEL = "claude-3-5-sonnet-20241022"

print("Client ready.")

## Step 2 — Ask a Question the Model Cannot Know

**Naija One Bank** is a fictional Nigerian bank we'll use throughout this module. Its Flexi Save account details exist only in a policy document we haven't shown the model yet.

Let's ask Claude about it cold, with no context supplied.

In [ ]:
question = "What is the current interest rate on the Flexi Save savings account at Naija One Bank, and what is the minimum opening balance?"

response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    messages=[
        {"role": "user", "content": question}
    ]
)

print(response.content[0].text)

## Step 3 — Look Closely at That Answer

Read the output above carefully. You'll typically see one of two things:

- **Honest uncertainty** — Claude says it has no information about "Naija One Bank" specifically, because it's a fictional institution the model was never trained on
- **Hallucination** — with a real bank name, or a more general phrasing, models will often confidently invent a plausible-sounding rate and minimum balance instead of admitting they don't know

Either outcome proves the same point: **the model has no way to answer this correctly, because the answer was never in its training data.** No amount of clever prompting fixes this — the model simply doesn't have the information.

The only fix is to give it the information.

## Step 4 — Here's the Actual Policy Document

Below is the real (fictional) Naija One Bank policy text. In a production RAG system, this text would live in a knowledge base and get found automatically by a retriever. For now, we'll skip retrieval entirely and just paste it in by hand — that's the whole point of this first demo.

In [ ]:
policy_document = """
Naija One Bank — Flexi Save Account Policy (Effective 2026)

The Flexi Save account is Naija One Bank's flagship savings product for individual
customers. Key terms:

- Interest rate: 4.2% per annum, calculated daily and credited monthly
- Minimum opening balance: NGN 5,000
- Minimum balance to earn interest: NGN 1,000
- Withdrawal limit: 3 free withdrawals per month, NGN 500 fee thereafter
- No monthly maintenance fee if minimum balance is maintained
"""

print(policy_document)

## Step 5 — Ask the Same Question, With Context Supplied

Now we build a new prompt that includes the policy document **before** the question, and instruct the model to answer only from what's provided.

In [ ]:
augmented_prompt = f"""Use the following document to answer the question. Only use information from the document below.

DOCUMENT:
{policy_document}

QUESTION:
{question}
"""

response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    messages=[
        {"role": "user", "content": augmented_prompt}
    ]
)

print(response.content[0].text)

## Step 6 — Compare the Two Answers

Look at the two outputs side by side:

| | Without context (Step 2) | With context (Step 5) |
|--|--------------------------|------------------------|
| Interest rate | Guessed or refused | 4.2% per annum — correct |
| Minimum balance | Guessed or refused | NGN 5,000 — correct |
| Where did it come from? | The model's training data (or nowhere) | The document we supplied |

**Nothing about the model changed between these two calls.** Same model, same weights, same API. The only difference is that the second call was *given the right information before it had to answer.*

That's the entire idea behind RAG: **retrieve the right document, then generate the answer from it.**

## What's Next

In this demo, *we* found the relevant document and pasted it in by hand. In a real system with thousands of documents, that's not possible — you can't paste your entire knowledge base into every prompt.

The rest of this module builds the missing piece:

- **Topic 2** — loading and chunking documents so they're ready to search
- **Topic 3** — building a retriever that automatically finds the right chunk
- **Topic 5** — wiring that retriever into the LLM call you just built here

By the end, the manual "paste the policy in" step you just did will happen automatically, for any question, across any number of documents.